# 저장 모델 추론

앞선 학습에서 저장한 Transformer checkpoint·scaler·경보 기준을 불러와 무라벨 CSV를 점수화한다. 새 optimizer step은 수행하지 않는다.

실행 입력은 학습 결과와 `prepare_demo_csv.py`가 만드는 공개 D2 예제다. 코드 cell의 출력은 실행 시 생성한다. 별도 공정 데이터로 일반화 성능을 평가하는 실험과는 구분한다.


In [ ]:
from pathlib import Path  # 프로젝트의 checkpoint와 결과 파일을 같은 기준 경로에서 찾습니다.
import json  # 이전 실제 학습 JSON과 실제 추론 측정 manifest를 읽고 저장합니다.
import time  # 저장 모델을 불러와 새 CSV를 점수화한 실제 경과시간을 측정합니다.
import hashlib  # 실행 전후 checkpoint 파일이 바뀌지 않았음을 hash로 확인합니다.
import torch  # 현재 CUDA/GPU와 저장된 PyTorch 모델의 weight를 확인합니다.
import pandas as pd  # 무라벨 입력과 추론 queue를 표로 표시합니다.
import matplotlib.pyplot as plt  # 이전 실제 epoch 기록을 학습 곡선으로 보여줍니다.
from IPython.display import display  # 표와 metadata를 notebook output으로 남깁니다.
project_dir = Path.cwd()  # 실행 커널의 작업 위치를 프로젝트 기준점으로 잡습니다.
if not (project_dir / 'st_awfd_causal_gpt_multihorizon.py').exists():  # 다른 폴더에서 잘못된 모델을 읽는 일을 막습니다.
    raise FileNotFoundError('Run this notebook from the pe_yield_lab project folder.')  # 필요한 실행 위치를 알립니다.
result_dir = project_dir / 'results' / 'st_awfd' / 'causal_gpt_multihorizon'  # 기존 학습 기록 및 이번 시연 CSV를 저장할 폴더입니다.
checkpoint_path = project_dir / 'models' / 'st_awfd' / 'st_awfd_strict_causal_multihorizon_gpt.pt'  # 학습 완료한 best epoch weight 경로입니다.
checkpoint_hash_before = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()  # 추론 전에 저장 weight 파일의 지문을 남깁니다.
print('Project:', project_dir)  # 실제로 사용한 프로젝트 폴더를 표시합니다.
print('PyTorch:', torch.__version__)  # 현재 실행 라이브러리를 기록합니다.
print('CUDA available:', torch.cuda.is_available())  # 실제 GPU 사용 가능 여부를 출력합니다.
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')  # 3070 Ti에서 실행하는지 눈으로 확인합니다.
print('New neural training epochs in this notebook:', 0)  # 이번 실행이 학습 없이 추론하는 노트북임을 명시합니다.

## 저장된 학습 이력

모델이 남긴 JSON과 epoch CSV에서 사용 장치, 선택 epoch와 학습 곡선을 읽는다. Gaussian NLL은 예측 분포와 관측값의 차이이며, 정상 예측의 likelihood를 나타낸다. 그래프는 저장 기록이고 이번 cell의 새 학습 결과는 아니다.


In [ ]:
summary_path = result_dir / 'strict_causal_gpt_run_summary.json'  # 실제 저장된 GPU run 요약을 읽습니다.
training_summary = json.loads(summary_path.read_text(encoding='utf-8'))  # best epoch, GPU 및 검증 지표를 원래 기록에서 가져옵니다.
history = pd.read_csv(result_dir / 'strict_causal_gpt_training_history.csv')  # 실제 optimizer epoch마다 기록한 loss CSV입니다.
selected_epoch = int(training_summary['training']['selected_epoch'])  # 정상 validation으로 선택했던 checkpoint epoch를 찾습니다.
selected_row = history.loc[history['epoch'] == selected_epoch].iloc[0]  # 그 checkpoint에 해당하는 실제 loss 행을 얻습니다.
first_validation_nll = float(history.iloc[0]['validation_multi_horizon_gaussian_nll'])  # epoch 1의 정상 forecast loss를 표시합니다.
best_validation_nll = float(selected_row['validation_multi_horizon_gaussian_nll'])  # best epoch의 정상 forecast loss를 표시합니다.
display(pd.DataFrame([training_summary['training']]))  # 이전 실제 학습의 장치·표본 수·완료/선택 epoch를 보여줍니다.
print(f'Actual recorded history: {len(history)} epochs | selected {selected_epoch} | normal-val {first_validation_nll:.6f} -> {best_validation_nll:.6f}')  # curve의 시작과 best 값을 실제 CSV에서 출력합니다.
figure, axis = plt.subplots(figsize=(10, 4.5))  # notebook에 보기 쉬운 학습 기록 그림을 만듭니다.
axis.plot(history['epoch'], history['train_multi_horizon_gaussian_nll'], label='Recorded training Gaussian NLL')  # 이전 실제 정상 fit loss를 그립니다.
axis.plot(history['epoch'], history['validation_multi_horizon_gaussian_nll'], label='Recorded normal-validation Gaussian NLL')  # 이전 실제 정상 validation loss를 그립니다.
axis.axvline(selected_epoch, color='tab:red', linestyle='--', label=f'Selected checkpoint: epoch {selected_epoch}')  # 추론에 쓰는 best weight가 어느 epoch인지 표시합니다.
axis.scatter([selected_epoch], [best_validation_nll], color='tab:red', zorder=3)  # best 정상 validation 지점을 강조합니다.
axis.set(xlabel='Actual recorded epoch', ylabel='Gaussian NLL', title='Previously completed RTX 3070 Ti training — loaded history, no new training')  # 새 학습으로 오해하지 않도록 제목을 명확히 씁니다.
axis.grid(alpha=0.25)  # epoch별 변화와 loss 수준을 읽기 쉽게 합니다.
axis.legend()  # training과 정상 validation의 구분을 표시합니다.
figure.tight_layout()  # 표제와 축이 잘리지 않게 여백을 정리합니다.
figure.savefig(result_dir / 'notebook_loaded_actual_training_history.png', dpi=160)  # 실제 기록을 그린 그림도 별도 파일로 보관합니다.
plt.show()  # 실제 curve를 notebook output에 저장합니다.
plt.close(figure)  # 뒤의 실행 셀에서 그림이 중복 출력되는 일을 줄입니다.

## Weight와 구조

각 시점의 센서 20개가 하나의 수치 token이다. Kernel 5의 과거 방향 Conv1d, causal attention 3개 block·4개 head, hidden 64·FFN 128을 사용한다. 네 horizon의 Gaussian 평균과 log-sigma를 출력한다.

Checkpoint를 복원하고 eval/no_grad로 추론한다. 완료 trace의 관측값과 예측 분포를 비교해 lot별 NLL을 계산한다.


In [ ]:
import st_awfd_causal_gpt_multihorizon as experiment  # 학습과 동일한 구조 및 저장 모델의 CSV 추론 API를 가져옵니다.
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)  # 자체 생성 checkpoint의 tensor와 기본 metadata만 읽습니다.
model_shell = experiment.CausalMultiHorizonGPT(int(checkpoint['feature_count']), int(checkpoint['sequence_length']))  # 저장 구조와 같은 신경망 틀을 만들어 parameter 수를 확인합니다.
model_shell.load_state_dict(checkpoint['state_dict'], strict=True)  # 실제 best epoch의 학습된 weight가 구조에 맞게 복원되는지 확인합니다.
model_shell.eval()  # 구조 확인에서도 dropout을 꺼 고정 추론 모드로 둡니다.
parameter_count = sum(parameter.numel() for parameter in model_shell.parameters())  # checkpoint가 사용하는 학습 parameter 수를 계산합니다.
architecture_record = {'parameter_count': parameter_count, 'feature_count': checkpoint['feature_count'], 'sequence_length': checkpoint['sequence_length'], 'forecast_horizons': checkpoint['horizons'], 'selected_epoch': checkpoint['best_epoch'], 'saved_normal_reference_limit': checkpoint['normal_reference_limit'], 'architecture': checkpoint['architecture']}  # 실제 저장 모델의 크기와 목적을 한 표로 보여줍니다.
display(pd.DataFrame([architecture_record]))  # 단순한 모델 이름 대신 실제 입력·horizon·parameter를 확인합니다.
display(pd.Series(training_summary['causality_audit'], name='saved_tensor_causality_audit'))  # 과거 실제 future-perturbation 검사의 수치를 읽어 보여줍니다.
print('Saved tensor-level causal audit; preprocessing still uses completed whole-step traces.')  # raw live stream의 online causal 보장으로 확대 해석하지 않도록 설명합니다.

## 무라벨 lot 추론

예제 CSV의 20개 lot에서 전체 Step 1 trace를 읽는다. 정답과 split 열은 입력하지 않는다. 고정 threshold 초과 여부와 상위 10% 검토 후보를 각각 기록하고, 실행 전후 checkpoint hash가 같은지 확인한다.


In [ ]:
demo_source = project_dir / 'results' / 'st_awfd' / 'st_awfd_label_free_demo_input.csv'  # 공식 holdout에서 정답을 제거한 기존 실행 시연 입력입니다.
demo_rows = pd.read_csv(demo_source)  # trace 행을 읽되 모델 fit이나 평가 label을 만들지 않습니다.
demo_ids = demo_rows['MaterialID'].drop_duplicates().head(20)  # 서로 다른 MaterialID group 20개를 고릅니다.
if len(demo_ids) != 20:  # 요청한 20개 group이 실제로 존재하는지 확인합니다.
    raise ValueError(f'Expected 20 demo MaterialID groups, found {len(demo_ids)}.')  # 표본 부족을 숨기지 않고 알립니다.
sensor_columns = [f'feature_{index}' for index in range(1, 21)]  # 학습 때의 익명 sensor 순서를 그대로 씁니다.
input_columns = ['MaterialID', 'StepID', 'duration_ms', *sensor_columns]  # 추론에 필요한 label 없는 필드만 선택합니다.
demo_20_rows = demo_rows.loc[demo_rows['MaterialID'].isin(demo_ids), input_columns].copy()  # 일부 행이 아니라 선택한 group의 전체 trace를 보존합니다.
demo_20_input = result_dir / 'notebook_demo_20_lots_input.csv'  # 이번 실제 notebook 추론에 쓴 label-free 입력 파일을 남깁니다.
demo_20_rows.to_csv(demo_20_input, index=False, encoding='utf-8-sig')  # 입력을 재현할 수 있도록 CSV로 저장합니다.
print('Actual input:', len(demo_20_rows), 'trace rows |', demo_20_rows['MaterialID'].nunique(), 'MaterialID groups | target/is_test excluded')  # 실제 처리할 입력 크기와 label 제거를 확인합니다.
if torch.cuda.is_available():  # CUDA의 비동기 작업까지 포함한 경과시간을 재기 전에 동기화합니다.
    torch.cuda.synchronize()  # 앞선 GPU 작업이 측정시간에 섞이지 않게 합니다.
inference_started = time.perf_counter()  # 실제 API 호출 직전 시간을 기록합니다.
inference_queue = experiment.score_new_csv(demo_20_input, checkpoint_path=checkpoint_path, review_fraction=0.10)  # 저장 모델을 복원해 실제 무라벨 20개 trace의 CUDA 추론을 수행합니다.
if torch.cuda.is_available():  # 추론의 GPU 연산이 실제 완료됐는지 동기화합니다.
    torch.cuda.synchronize()  # 결과 전달까지 GPU 작업이 끝난 시점으로 측정합니다.
inference_seconds = time.perf_counter() - inference_started  # CSV 로드·모델 복원·전처리·추론을 포함한 API 실행시간입니다.
queue_attrs = dict(inference_queue.attrs)  # 함수가 기록한 실제 device와 checkpoint 정보를 보관합니다.
queue_columns = ['MaterialID', 'anomaly_score', 'normal_reference_limit', 'above_normal_reference_limit', 'review_rank', 'review_candidate_at_capacity', 'leading_anonymous_feature', 'largest_surprise_horizon']  # PE 검토에 직접 읽을 출력 필드를 묶습니다.
display(inference_queue[queue_columns])  # 실제 추론 결과 20개를 notebook output에 남깁니다.
display(pd.Series(queue_attrs, name='actual_inference_metadata'))  # 실제 CUDA 장치 및 사용한 epoch와 전처리 범위를 표시합니다.
print(f'Actual inference API elapsed: {inference_seconds:.6f} seconds | new neural training epochs: 0')  # 수신한 결과가 실제 실행에서 나왔음을 시간과 함께 기록합니다.
output_csv = result_dir / 'notebook_executed_20_lot_review_queue.csv'  # 이번 시연의 결과를 기존 전체 holdout 결과와 다른 파일명으로 저장합니다.
inference_queue.to_csv(output_csv, index=False, encoding='utf-8-sig')  # Excel에서 다시 정리할 수 있는 실제 queue CSV를 저장합니다.
checkpoint_hash_after = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()  # 추론 후 checkpoint의 hash를 다시 계산합니다.
if checkpoint_hash_after != checkpoint_hash_before:  # 추론 중 저장 weight가 변경되는 예상 밖의 상황을 탐지합니다.
    raise RuntimeError('Checkpoint changed during inference-only notebook execution.')  # 추론만 했다는 주장을 검증하지 못하면 실행을 중단합니다.
inference_manifest = {'input_group_count': int(demo_20_rows['MaterialID'].nunique()), 'input_row_count': len(demo_20_rows), 'output_group_count': len(inference_queue), 'actual_inference_api_seconds': inference_seconds, 'new_neural_training_epochs': 0, 'checkpoint_unchanged': checkpoint_hash_after == checkpoint_hash_before, 'checkpoint_sha256': checkpoint_hash_after, 'queue_capacity_count': int(inference_queue['review_candidate_at_capacity'].sum()), 'above_fixed_limit_count': int(inference_queue['above_normal_reference_limit'].sum()), 'metadata': queue_attrs, 'input_csv': str(demo_20_input), 'output_csv': str(output_csv), 'note': 'Executed label-free demo inference; not independent external fab performance or measured yield uplift.'}  # 실행 장치·시간·불변 weight 및 출력 위치를 남깁니다.
(result_dir / 'notebook_executed_inference_manifest.json').write_text(json.dumps(inference_manifest, ensure_ascii=False, indent=2), encoding='utf-8')  # 이번 실제 notebook 실행을 JSON으로도 감사할 수 있게 합니다.
display(pd.Series(inference_manifest, name='actual_execution_record'))  # 결과 파일과 실제 측정 기록을 notebook에서 확인합니다.
print('Saved output CSV:', output_csv)  # Excel 또는 별도 분석에서 다시 열 파일 위치를 표시합니다.

## 공분산 비교 기록

별도 실행에서 저장한 공분산 점수 비교가 있으면 함께 표시한다. 이 cell은 공분산이나 신경망을 다시 fit하지 않는다. 정상 오경보와 순위 성능의 변화를 같은 표에서 확인한다.


In [ ]:
covariance_dir = project_dir / 'results' / 'st_awfd' / 'gpt_covariance_exploratory'  # 기존 별도 탐색의 저장 결과 폴더입니다.
if (covariance_dir / 'covariance_exploratory_summary.json').is_file():
    covariance_summary = json.loads((covariance_dir / 'covariance_exploratory_summary.json').read_text(encoding='utf-8'))  # 실제 별도 탐색의 정책·checkpoint 불변·다중 지표를 읽습니다.
    covariance_comparison = pd.read_csv(covariance_dir / 'model_comparison_exploratory.csv')  # 기존 NLL·공분산 score·고전 baseline의 비교표를 읽습니다.
    comparison_columns = ['model', 'holdout_average_precision', 'holdout_roc_auc', 'holdout_normal_false_alarm_rate', 'holdout_anomaly_recall_at_normal_limit', 'holdout_precision_at_normal_limit', 'holdout_f1_at_normal_limit', 'top_review_anomaly_recall', 'top_review_precision']  # 순위·오경보·검출·capacity를 함께 비교합니다.
    display(covariance_comparison[comparison_columns])  # 저장된 실제 탐색의 장점과 손실을 같은 표로 보여줍니다.
    print('Loaded earlier covariance audit | new neural epochs there:', covariance_summary['new_neural_training_epochs'], '| checkpoint unchanged:', covariance_summary['checkpoint_unchanged'])  # 이번 셀은 재계산 없이 기존 기록을 읽는다는 의미입니다.
    print('This notebook reran only label-free inference; covariance fitting and neural training were not rerun.')  # 셀 출력이 새 학습 결과처럼 보이지 않도록 기록합니다.
else:
    print('저장된 공분산 비교 결과 없음')
